**Auteur(s)** : Cheikhou Akhmed KANE

**Description** : Mise en place `reglesDeDecision.csv`

# Création du Fichier de Règles de Décision

## 1. Description du projet

Ce notebook a pour objectif de générer le fichier `reglesDeDecisions.csv` pour le territoire de Sasseme.

La méthode consiste à partir d'un fichier existant (`reglesDeDecisionsextended.csv`), à le simplifier en ne gardant que les 6 itinéraires techniques (ITK) pertinents, puis à paramétrer finement les opérations techniques (`PREPA`, `SEMIS`, `RECOLTE`) via un dictionnaire de configuration.

---
## 2. Objectifs

* Charger le fichier de règles de décision source.
* Créer une nouvelle structure avec seulement 6 ITK.
* Remplir l'en-tête (12 premières lignes) pour définir ces 6 ITK.
* Initialiser toutes les opérations techniques à "désactivé" (`NA` ou `N`).
* Activer et paramétrer spécifiquement les opérations `PREPA`, `SEMIS` et `RECOLTE` à l'aide d'un dictionnaire.
* Sauvegarder les fichiers finaux.

---
## 3. Méthodologie

Le processus est décomposé en plusieurs fonctions utilitaires pour plus de clarté et de reproductibilité :

* **`creer_structure_itk()`** : Crée la "coquille" vide du DataFrame final, en conservant les 3 colonnes de description et en ajoutant 6 colonnes vides pour nos nouveaux ITK.
* **`ajouter_lignes_semis()`** : Vérifie la présence des paramètres de semis liés à la pluie et les ajoute au DataFrame si nécessaire pour garantir la robustesse du script.
* **`remplir_entete_itk()`** : Remplit les 12 premières lignes (l'en-tête) de chaque ITK en se basant sur les règles définies (ID de l'espèce, du précédent, etc.).
* **`initialiser_operations_a_na()`** : Crée une "page blanche" en désactivant par défaut toutes les opérations techniques (valeurs `NA` ou `N`) après l'en-tête.
* **`appliquer_parametres_specifiques()`** : Active et configure finement les opérations `PREPA`, `SEMIS`, et `RECOLTE` en appliquant les valeurs d'un dictionnaire de configuration.

---
## 4. Fichiers en Entrée et en Sortie

### 4.1. Fichier en Entrée
* **Règles de Décision (Source)** : `data/ITK/csv/raw/reglesDeDecisionsextended.csv`

### 4.2. Fichiers en Sortie
* **Règles de Décision (Sasseme)** : `includes_sassemeV1/modeleAgricole/culture/reglesDeDecisions.csv`
---

In [1]:
import pandas as pd
from pathlib import Path

In [11]:
base_dir = Path.cwd().parent.resolve()

# Fichier en entrée
input_regles_path = base_dir / "data" / "ITK" / "csv" / "raw" / "reglesDeDecisionsextended.csv"

# Fichiers en sortie
output_regles_path = base_dir / "includes_sassemeV1" / "modeleAgricole" / "culture" / "reglesDeDecision.csv"
output_ferti_path = base_dir / "includes_sassemeV1" / "modeleAgricole" / "culture" / "reglesDeDecisionFertilisation.csv"

In [12]:
# Liste de nos 6 ITK cibles
itk_sasseme = [
    'arachide_precMil',
    'arachide_precJachere',
    'jachere_precMil',
    'jachere_precArachide',
    'mil_precArachide',
    'mil_precMil'
]

In [13]:
try:
    df_source = pd.read_csv(input_regles_path, sep=';')
    print("✅ Fichier source des règles de décision chargé avec succès.")
    print(f"   -> Contient {df_source.shape[0]} règles et {df_source.shape[1]} colonnes (ITK).")

except FileNotFoundError:
    print(f"🚨 ERREUR : Fichier non trouvé. Vérifiez le chemin : {input_regles_path}")
except Exception as e:
    print(f"🚨 Une erreur est survenue : {e}")

✅ Fichier source des règles de décision chargé avec succès.
   -> Contient 270 règles et 89 colonnes (ITK).


In [14]:
def creer_structure_itk(df_source, nouvelle_liste_itk):
    """
    Crée un nouveau DataFrame avec les 3 colonnes de base spécifiées
    et de nouvelles colonnes vides pour une liste d'ITK donnée.

    Args:
        df_source (pd.DataFrame): Le DataFrame original des règles de décision.
        nouvelle_liste_itk (list): La liste des noms pour les nouveaux ITK.

    Returns:
        pd.DataFrame: Le nouveau DataFrame avec la structure cible.
    """
    # 1. Définir et garder les 3 colonnes de base exactes
    colonnes_base = ["NOM_ITK_AFFICHAGE", "X.", "gel"]
    df_cible = df_source[colonnes_base].copy()

    # 2. Ajouter les nouvelles colonnes d'ITK (vides)
    for itk in nouvelle_liste_itk:
        df_cible[itk] = pd.NA

    print("✅ Structure du nouveau DataFrame créée avec succès.")
    return df_cible

# --- Exécution ---
# On applique la fonction pour créer notre DataFrame de travail
df_sasseme = creer_structure_itk(df_source, itk_sasseme)

# --- VÉRIFICATION ---
print(f"   -> Le nouveau DataFrame contient {df_sasseme.shape[0]} lignes et {df_sasseme.shape[1]} colonnes.")
print("\nAperçu de la structure (les 6 dernières colonnes sont vides) :")
display(df_sasseme.head())

✅ Structure du nouveau DataFrame créée avec succès.
   -> Le nouveau DataFrame contient 270 lignes et 9 colonnes.

Aperçu de la structure (les 6 dernières colonnes sont vides) :


,NOM_ITK_AFFICHAGE,X.,gel,arachide_precMil,arachide_precJachere,jachere_precMil,jachere_precArachide,mil_precArachide,mil_precMil
0,ID_ITK,[NA],gel_tous_tous_NA,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,IDS_SDCS,[NA],*,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,IDS_SDCS_CLASS,[NA],*,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,ID_ESPECE,[NA],gel,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,MATERIEL,[NA],*,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [15]:
def ajouter_lignes_semis(df):
    """
    Vérifie si les lignes de paramètres de pluie pour le semis existent,
    et les ajoute si elles sont manquantes.
    """
    # Noms des nouvelles lignes
    ligne_cumul_pluie = 'SEMIS_CUMUL_PLUIE'
    ligne_n_j_cumul_pluie = 'SEMIS_N_J_CUMUL_PLUIE'

    # Vérifier si la première ligne existe déjà
    if ligne_cumul_pluie not in df['NOM_ITK_AFFICHAGE'].values:
        print(f"-> La ligne '{ligne_cumul_pluie}' est manquante. Ajout en cours...")
        
        # 1. Trouver où insérer les nouvelles lignes : après la dernière ligne "SEMIS_"
        lignes_semis = df[df['NOM_ITK_AFFICHAGE'].str.startswith('SEMIS_', na=False)]
        if not lignes_semis.empty:
            index_insertion = lignes_semis.index[-1] + 1
        else:
            # Sécurité si aucune ligne SEMIS n'existait
            index_insertion = len(df) 
            
        # 2. Créer les nouvelles lignes sous forme de DataFrames
        nouvelle_ligne1 = pd.DataFrame({
            'NOM_ITK_AFFICHAGE': [ligne_cumul_pluie],
            'X.': ['[mm]'],
            'gel': ['NA']
        })
        nouvelle_ligne2 = pd.DataFrame({
            'NOM_ITK_AFFICHAGE': [ligne_n_j_cumul_pluie],
            'X.': ['[jour]'],
            'gel': ['NA']
        })

        # 3. Insérer les lignes au bon endroit
        df_partie1 = df.iloc[:index_insertion]
        df_partie2 = df.iloc[index_insertion:]
        df = pd.concat([df_partie1, nouvelle_ligne1, nouvelle_ligne2, df_partie2], ignore_index=True)
        
        print("   -> Lignes ajoutées avec succès.")
    else:
        print(f"-> Les lignes de paramètres de pluie pour le semis existent déjà.")
        
    return df

df_sasseme = ajouter_lignes_semis(df_sasseme)

-> La ligne 'SEMIS_CUMUL_PLUIE' est manquante. Ajout en cours...
   -> Lignes ajoutées avec succès.


In [16]:
def remplir_entete_itk(df_cible):
    """
    Remplit les 12 lignes d'en-tête pour chaque nouvel ITK en se basant sur
    le nom de la colonne de l'ITK.

    Args:
        df_cible (pd.DataFrame): Le DataFrame avec la structure vide.

    Returns:
        pd.DataFrame: Le DataFrame avec l'en-tête rempli.
    """
    # 1. Identifier les colonnes des ITK (de la 4ème à la fin)
    colonnes_itk = df_cible.columns[3:]

    # 2. Boucler sur chaque colonne d'ITK
    for itk in colonnes_itk:
        # Extraire la culture et le précédent du nom de l'ITK
        try:
            culture, precedent = itk.split('_prec')
            # On garde le nom en minuscules
            precedent = precedent.lower() 
        except ValueError:
            print(f"ATTENTION : Le nom de l'ITK '{itk}' ne suit pas le format 'culture_precPrecedent'.")
            continue

        # 3. Appliquer les règles pour chaque ligne de l'en-tête
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'NOM_ITK_AFFICHAGE', itk] = itk
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'ID_ITK', itk] = itk
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'IDS_SDCS', itk] = '*'
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'IDS_SDCS_CLASS', itk] = '*'
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'ID_ESPECE', itk] = culture
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'MATERIEL', itk] = 'NA'
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'ID_PREC', itk] = precedent
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'ZONE_PEDO', itk] = ''
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'ZONE_PEDO_CLASS', itk] = 'NA'
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'TYPE_EXPL', itk] = 'all'
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'CLIMAT', itk] = ''
        df_cible.loc[df_cible['NOM_ITK_AFFICHAGE'] == 'IS_CULTURE_HIVER', itk] = 'N'

    print("✅ En-tête des 6 ITK rempli avec succès.")
    return df_cible

# --- Exécution ---
# On applique la fonction sur notre DataFrame de travail
df_sasseme = remplir_entete_itk(df_sasseme)

# --- VÉRIFICATION ---
# On affiche les 12 premières lignes pour voir le résultat
print("\nAperçu de l'en-tête rempli :")
display(df_sasseme.head(12))

✅ En-tête des 6 ITK rempli avec succès.

Aperçu de l'en-tête rempli :


,NOM_ITK_AFFICHAGE,X.,gel,arachide_precMil,arachide_precJachere,jachere_precMil,jachere_precArachide,mil_precArachide,mil_precMil
0,ID_ITK,[NA],gel_tous_tous_NA,arachide_precMil,arachide_precJachere,jachere_precMil,jachere_precArachide,mil_precArachide,mil_precMil
1,IDS_SDCS,[NA],*,*,*,*,*,*,*
2,IDS_SDCS_CLASS,[NA],*,*,*,*,*,*,*
3,ID_ESPECE,[NA],gel,arachide,arachide,jachere,jachere,mil,mil
4,MATERIEL,[NA],*,NA,NA,NA,NA,NA,NA
5,ID_PREC,[NA],*,mil,jachere,mil,arachide,arachide,mil
6,ZONE_PEDO,[NA],*,,,,,,
7,ZONE_PEDO_CLASS,[NA],tous,NA,NA,NA,NA,NA,NA
8,TYPE_EXPL,[NA],all,all,all,all,all,all,all
9,CLIMAT,[NA],NaN,,,,,,


In [17]:
def initialiser_operations_a_na(df_cible):
    """
    Initialise toutes les opérations techniques (lignes après l'en-tête)
    à un état "désactivé" par défaut ('NA' ou 'N').

    Args:
        df_cible (pd.DataFrame): Le DataFrame avec l'en-tête déjà rempli.

    Returns:
        pd.DataFrame: Le DataFrame avec toutes les opérations initialisées.
    """
    # 1. Identifier les colonnes des ITK (de la 4ème à la fin)
    colonnes_itk = df_cible.columns[3:]
    
    # 2. Remplir toutes les cellules d'opérations (après la 12ème ligne) avec NA
    #    L'index 12 correspond à la 13ème ligne.
    df_cible.loc[12:, colonnes_itk] = 'NA'
    
    # 3. Mettre à 'N' (Non) toutes les lignes qui activent une opération (celles qui commencent par 'IS_')
    #    On s'assure de ne pas toucher à la ligne 'IS_CULTURE_HIVER' qui est dans l'en-tête.
    lignes_is = (df_cible['NOM_ITK_AFFICHAGE'].str.startswith('IS_', na=False)) & (df_cible.index > 11)
    df_cible.loc[lignes_is, colonnes_itk] = 'N'
    
    print("✅ Toutes les opérations techniques ont été initialisées à 'NA' ou 'N'.")
    return df_cible

# --- Exécution ---
# On applique la fonction sur notre DataFrame de travail
df_sasseme = initialiser_operations_a_na(df_sasseme)

# --- VÉRIFICATION ---
# On affiche un extrait du DataFrame après la ligne 12 pour voir le résultat
print("\nAperçu de la 'page blanche' pour les opérations :")
display(df_sasseme.loc[12:25]) # Affiche les lignes de 13 à 26

✅ Toutes les opérations techniques ont été initialisées à 'NA' ou 'N'.

Aperçu de la 'page blanche' pour les opérations :


,NOM_ITK_AFFICHAGE,X.,gel,arachide_precMil,arachide_precJachere,jachere_precMil,jachere_precArachide,mil_precArachide,mil_precMil
12,PREPA_PASSAGES,[NA],NaN,NA,NA,NA,NA,NA,NA
13,PREPA_OUTIL,[nom],NaN,NA,NA,NA,NA,NA,NA
14,PREPA_AGRIW,[bool],NaN,NA,NA,NA,NA,NA,NA
15,PREPA_NB_SOUS_PERIODES,[NA],NaN,NA,NA,NA,NA,NA,NA
16,PREPA_TEMPS,[Ha/h],NaN,NA,NA,NA,NA,NA,NA
17,PREPA_DEBUT,[jour],NaN,NA,NA,NA,NA,NA,NA
18,PREPA_FIN,[jour],NaN,NA,NA,NA,NA,NA,NA
19,PREPA_JOURS_P-ETP_MIN,[jour],NaN,NA,NA,NA,NA,NA,NA
20,PREPA_P-ETP_MIN,[mm],NaN,NA,NA,NA,NA,NA,NA
21,PREPA_JOURS_PLUIE,[jour],NaN,NA,NA,NA,NA,NA,NA


In [21]:
# --- Dictionnaire de configuration pour Sasseme ---

configuration_sasseme = {
    'PREPA': {
        'IS_PREPA': {'valeur': 'O'},
        'PREPA_TEMPS': {'valeur': 0.0417},
        'PREPA_DEBUT': {'valeur': 121},
        'PREPA_FIN': {'valeur': 151}
    },
    'SEMIS': {
        'IS_SEMIS': {'valeur': 'O'},
        'SEMIS_DEBUT': {'valeur': 152},
        'SEMIS_FIN': {'valeur': 181},
        # Paramètres spécifiques à l'arachide
        'SEMIS_CUMUL_PLUIE': {
            'valeur': 20,
            'condition': {'ligne_condition': 'ID_ESPECE', 'valeur_attendue': 'arachide'}
        },
        'SEMIS_N_J_CUMUL_PLUIE': {
            'valeur': 1,
            'condition': {'ligne_condition': 'ID_ESPECE', 'valeur_attendue': 'arachide'}
        }
    },
    'RECOLTE': {
        'IS_RECOLTE': {'valeur': 'O'},
        'RECOLTE_TEMPS': {'valeur': 0.0179},
        'RECOLTE_DEBUT': {'valeur': 258},
        'RECOLTE_FIN': {'valeur': 288}
    }
}

In [22]:
def appliquer_parametres_specifiques(df_cible, configuration):
    """
    Active et paramètre des opérations spécifiques dans le DataFrame des règles
    en se basant sur un dictionnaire de configuration.
    """
    colonnes_itk = df_cible.columns[3:]

    for operation, params in configuration.items():
        for nom_param, details in params.items():
            valeur = details['valeur']
            condition = details.get('condition')

            ligne_index = df_cible.index[df_cible['NOM_ITK_AFFICHAGE'] == nom_param]
            if ligne_index.empty:
                print(f"-> AVERTISSEMENT : Le paramètre '{nom_param}' n'a pas été trouvé dans la table.")
                continue

            if not condition:
                df_cible.loc[ligne_index, colonnes_itk] = valeur
            else:
                # Utilise la clé 'ligne_condition'
                nom_ligne_condition = condition['ligne_condition']
                valeur_attendue = condition['valeur_attendue']
                
                ligne_condition_index = df_cible.index[df_cible['NOM_ITK_AFFICHAGE'] == nom_ligne_condition]
                
                colonnes_a_modifier = [
                    col for col in colonnes_itk 
                    if df_cible.loc[ligne_condition_index, col].iloc[0] == valeur_attendue
                ]
                
                df_cible.loc[ligne_index, colonnes_a_modifier] = valeur
    
    print("✅ Paramètres spécifiques appliqués avec succès.")
    return df_cible

# --- Exécution ---
df_sasseme = appliquer_parametres_specifiques(df_sasseme, configuration_sasseme)

# --- VÉRIFICATION ---
print("\nAperçu de quelques paramètres modifiés :")
parametres_a_verifier = [
    'IS_PREPA', 'PREPA_TEMPS', 'PREPA_DEBUT', 'PREPA_FIN',
    'IS_SEMIS', 'SEMIS_DEBUT', 'SEMIS_FIN', 'SEMIS_CUMUL_PLUIE', 'SEMIS_N_J_CUMUL_PLUIE',
    'IS_RECOLTE', 'RECOLTE_TEMPS', 'RECOLTE_DEBUT', 'RECOLTE_FIN',
    'IS_BINAGE'
]
display(df_sasseme[df_sasseme['NOM_ITK_AFFICHAGE'].isin(parametres_a_verifier)])

✅ Paramètres spécifiques appliqués avec succès.

Aperçu de quelques paramètres modifiés :


,NOM_ITK_AFFICHAGE,X.,gel,arachide_precMil,arachide_precJachere,jachere_precMil,jachere_precArachide,mil_precArachide,mil_precMil
11,IS_PREPA,[NA],N,O,O,O,O,O,O
16,PREPA_TEMPS,[Ha/h],NaN,0.0417,0.0417,0.0417,0.0417,0.0417,0.0417
17,PREPA_DEBUT,[jour],NaN,121,121,121,121,121,121
18,PREPA_FIN,[jour],NaN,151,151,151,151,151,151
57,IS_SEMIS,[NA],O,O,O,O,O,O,O
63,SEMIS_DEBUT,[jour],10,152,152,152,152,152,152
64,SEMIS_FIN,[jour],20,181,181,181,181,181,181
76,SEMIS_CUMUL_PLUIE,[mm],NA,20,20,NA,NA,NA,NA
77,SEMIS_N_J_CUMUL_PLUIE,[jour],NA,1,1,NA,NA,NA,NA
78,IS_BINAGE,[NA],N,N,N,N,N,N,N


In [23]:
# Sauvegarder le fichier principal des règles de décision
try:
    output_regles_path.parent.mkdir(parents=True, exist_ok=True)
    df_sasseme.to_csv(output_regles_path, sep=';', index=False)
    
    print("✅ Fichier 'reglesDeDecision.csv' sauvegardé avec succès.")
    print(f"   -> Emplacement : {output_regles_path}")

except Exception as e:
    print(f"🚨 ERREUR lors de la sauvegarde de reglesDeDecision.csv : {e}")

✅ Fichier 'reglesDeDecision.csv' sauvegardé avec succès.
   -> Emplacement : C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\includes_sassemeV1\modeleAgricole\culture\reglesDeDecision.csv
